# utils

**Source:** `00_common/utils.py`  
**Purpose:** Databricks notebook auto-generated from framework Python module.


## Section 1: Additional module code and configuration

This cell handles: *Additional module code and configuration*


In [ ]:
"""Common utility helpers for Databricks ingestion framework."""

from __future__ import annotations

import hashlib
import json
import logging
import sys
import uuid
from datetime import datetime, timezone
from typing import Any, Dict


## Section 2: Define `utc_now_iso()` helper function

This cell handles: *Define `utc_now_iso()` helper function*


In [ ]:
def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


## Section 3: Define `new_load_id()` helper function

This cell handles: *Define `new_load_id()` helper function*


In [ ]:
def new_load_id() -> str:
    return str(uuid.uuid4())


## Section 4: Define `row_hash()` helper function

This cell handles: *Define `row_hash()` helper function*


In [ ]:
def row_hash(payload: Dict[str, Any]) -> str:
    canonical = json.dumps(payload, sort_keys=True, default=str)
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()


## Section 5: Define `require_keys()` helper function

This cell handles: *Define `require_keys()` helper function*


In [ ]:
def require_keys(payload: Dict[str, Any], required: list[str]) -> None:
    missing = [k for k in required if k not in payload]
    if missing:
        raise ValueError(f"Missing required keys: {missing}")


## Section 6: Define `truncate_str()` function with logic for processing

This cell handles: *Define `truncate_str()` function with logic for processing*


In [ ]:
def truncate_str(value: str, max_chars: int = 500) -> str:
    """Truncate a string to *max_chars*, replacing newlines for single-line output."""
    msg = str(value).replace("\n", " ").strip()
    return msg[:max_chars]


## Section 7: Define `_JsonFormatter` class

This cell handles: *Define `_JsonFormatter` class*


In [ ]:
class _JsonFormatter(logging.Formatter):
    """Emit each log record as a single-line JSON object."""


## Section 8: Define `format()` helper function

This cell handles: *Define `format()` helper function*


In [ ]:
    def format(self, record: logging.LogRecord) -> str:
        payload: Dict[str, Any] = {
            "ts": datetime.fromtimestamp(record.created, tz=timezone.utc).isoformat(),
            "level": record.levelname,
            "logger": record.name,
            "message": record.getMessage(),
        }
        if record.exc_info:
            payload["exc_info"] = self.formatException(record.exc_info)
        return json.dumps(payload, default=str)


## Section 9: Define `get_logger()` function with logic for processing

This cell handles: *Define `get_logger()` function with logic for processing*


In [ ]:
def get_logger(name: str = "ingestion_framework") -> logging.Logger:
    """Return a logger that writes structured JSON lines to stdout.

    Safe to call multiple times with the same name — returns the existing
    logger without adding duplicate handlers.
    """
    logger = logging.getLogger(name)
    if logger.handlers:
        return logger
    logger.setLevel(logging.DEBUG)
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(_JsonFormatter())
    logger.addHandler(handler)
    logger.propagate = False
    return logger
